In [ ]:
# 1. SETUP & CLONE LATEST REPOSITORY
%cd /content
! rm -rf Baseera
! git clone https://github.com/Hager-ali191/Baseera.git
%cd /content/Baseera

# 2. INSTALL DEPENDENCIES
! pip install -q -r backend/requirements.txt
! pip install -q -r NiceGUI/requirements.txt pillow groq httpx

# 3. LOAD ENVIRONMENT & GROQ API KEY
import os
import time
import subprocess
import gc
import re
import torch
from google.colab import userdata

# Load Groq key stored under Secrets (key icon) as 'siara'
try:
    groq_key = userdata.get("siara")
    os.environ["GROQ_API_KEY"] = groq_key
    os.environ["GROQ_MODEL"] = "llama-3.1-8b-instant"
    print("✓ GROQ_API_KEY successfully loaded from Colab secrets!")
except Exception as e:
    print("⚠ Warning: Could not load 'siara' secret from Colab. Error:", e)

# Fix typo in photo path references if needed
for rel_path in ["NiceGUI/pages/home.py", "NiceGUI/pages/demo.py"]:
    full_path = os.path.join("/content/Baseera", rel_path)
    if os.path.exists(full_path):
        with open(full_path, "r") as f:
            content = f.read()
        content = content.replace("MAIRAMM_PHOTO_PATH", "MARIAMM_PHOTO_PATH")
        content = content.replace("MAIRAM_PHOTO_PATH", "MAIRAMH_PHOTO_PATH")
        with open(full_path, "w") as f:
            f.write(content)

# Ensure NiceGUI is mounted to backend/main.py
main_py_path = "backend/main.py"
with open(main_py_path, "r") as f:
    main_content = f.read()

if "ui.run_with" not in main_content:
    mount_code = """
# --- NiceGUI Integration ---
import sys, os
sys.path.insert(0, os.path.abspath("../NiceGUI"))
from nicegui import ui
import pages.home, pages.about, pages.demo
ui.run_with(app, storage_secret="your_unique_secret_key")
"""
    with open(main_py_path, "a") as f:
        f.write(mount_code)

# 4. START UNIFIED FASTAPI + NICEGUI SERVER
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

! fuser -k 8000/tcp || true
time.sleep(1)

env = os.environ.copy()
env["PYTHONPATH"] = f"{os.getcwd()}/backend:{os.getcwd()}/NiceGUI:" + env.get("PYTHONPATH", "")
env["PYTHONUNBUFFERED"] = "1"

log_file = open("app.log", "w")
server = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"],
    cwd="backend",
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("Starting server on port 8000...")
time.sleep(6)

if server.poll() is None:
    print("✓ Combined FastAPI + NiceGUI server is UP on port 8000!")
else:
    print(f"❌ Server failed to start (exit code {server.poll()}). Logs:")
    ! cat app.log

# 5. EXPOSE WITH CLOUDFLARE TUNNEL
! if [ ! -f cloudflared ] ; then wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared ; fi

! pkill cloudflared || true
time.sleep(1)

tunnel_log = open("tunnel.log", "w")
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

print("Establishing Cloudflare tunnel...")
time.sleep(8)

with open("tunnel.log", "r") as f:
    logs = f.read()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
    if match:
        print("\n=======================================================")
        print("YOUR LIVE PROJECT URL IS:")
        print(match.group(0))
        print("=======================================================\n")

/content
Cloning into 'Baseera'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 142 (delta 42), reused 111 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 25.75 MiB | 19.29 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/Baseera
Updated NiceGUI/app.py with 'import os'
Successfully mounted NiceGUI into backend/main.py
